# Bronze layer

## 0) Dependências

In [0]:
# instalar Faker (persistente no cluster enquanto ativo)
%pip install Faker


## 1) Parâmetros e helpers

In [0]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Row
from datetime import datetime, timedelta
from faker import Faker
import random
import string

# ===== Parâmetros =====
CATALOG = "workshop_modelagem_databricks"  # o mesmo nome no Silver e no Gold. None = sem Unity Catalog
SCHEMA  = "bronze"        # esquema/database onde serão criadas as tabelas
SEED    = 42

N_CUSTOMERS   = 1000
N_PRODUCTS    = 500
N_ORDERS      = 5000
N_ORDER_ITEMS = 12000

# Percentuais de "problemas"
P_DUP_ORDERS                = 0.025   # ~2.5% order_id duplicados
P_NULL_CUSTOMER_IN_ORDERS   = 0.05    # ~5% customer_id nulos
P_STATUS_CASE_VARIATION     = 0.25
P_STRING_DATE_IN_ORDERS     = 0.50    # metade como string, metade como date coerente
P_STRING_NUMERIC_IN_FIELDS  = 0.15    # % de numéricos como string

P_DUP_PRODUCT_ID            = 0.03
P_INCONSISTENT_IS_ACTIVE    = 0.40
P_NULL_BRAND_SUBCATEGORY    = 0.10

P_CUSTOMER_INCONSISTENCY    = 0.20    # estados "SP", "sp", "São Paulo"
P_CUSTOMER_DUP_DIFF_UPDATE  = 0.10    # duplicar customer_id com last_update_date diferente
P_EMPTY_FIELDS_CUSTOMER     = 0.05

P_DUP_ORDERITEM_SAME_OP     = 0.05    # duplicar (order_id, product_id) com updated_at diferente
P_NULLS_DISCOUNT_PROMO      = 0.15

random.seed(SEED)
fake = Faker("pt_BR")
Faker.seed(SEED)

# ===== Nome totalmente qualificado de tabela =====
def fqtn(table):
    if CATALOG:
        return f"`{CATALOG}`.`{SCHEMA}`.`{table}`"
    else:
        return f"`{SCHEMA}`.`{table}`"

# ===== Criar schema/database =====
if CATALOG:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS `{CATALOG}`")
    spark.sql(f"CREATE SCHEMA  IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
else:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS `{SCHEMA}`")

# ===== Utilidades =====
STATUSES = ["delivered","shipped","processing","cancelled","returned"]

def random_status_inconsistent():
    s = random.choice(STATUSES)
    if random.random() < P_STATUS_CASE_VARIATION:
        # variações de capitalização
        choices = [s.upper(), s.capitalize(), s.lower()]
        s = random.choice(choices)
    return s

def random_date_between(days_back=365):
    base = datetime.utcnow()
    delta = timedelta(days=random.randint(0, days_back), seconds=random.randint(0, 86399))
    d = base - delta
    return d

def random_date_mixed_formats(dt):
    # retorna string em formatos variados ("/", "-", com/sem tempo)
    # ex.: "2025-10-25", "2025/10/25 14:33:20", "25/10/2025", etc.
    formats = [
        "%Y-%m-%d",
        "%Y/%m/%d",
        "%Y-%m-%d %H:%M:%S",
        "%d/%m/%Y",
        "%d-%m-%Y %H:%M:%S",
    ]
    return dt.strftime(random.choice(formats))

def maybe_stringify_number(x):
    # converte numérico para string em parte dos casos
    if random.random() < P_STRING_NUMERIC_IN_FIELDS:
        return f"{x}"
    return x

def maybe_null(val, p=0.1):
    return None if random.random() < p else val

def dirty_state(uf):
    # introduz inconsistências: "SP", "sp", "São Paulo"
    if random.random() < P_CUSTOMER_INCONSISTENCY:
        variants = [uf, uf.lower(), "São Paulo" if uf.upper()=="SP" else uf]
        return random.choice(variants)
    return uf

def random_bool_inconsistent():
    # "true", "1", "yes", True, False...
    opts = ["true","1","yes","false","0","no", True, False]
    if random.random() < P_INCONSISTENT_IS_ACTIVE:
        return random.choice(opts)
    return True

def alnum(n=8):
    return ''.join(random.choices(string.ascii_uppercase + string.digits, k=n))


## 2) Gravar as tabelas da Bronze

O nome no Unity Catalog tem três partes: `catalogo.schema.tabela`.

`bronze.orders` significa: catálogo atual + schema `bronze` + tabela `orders`.

Esta célula materializa quatro tabelas Delta no catálogo definido em `CATALOG`:

- `customers`
- `products`
- `orders`
- `order_items`

Todas as colunas ficam como `STRING`. A Bronze guarda o dado bruto (datas em vários formatos, números com vírgula, status com caixa misturada). Quem padroniza é a Silver.

As datas cabem nos últimos 50 dias de propósito: a Silver só lê uma janela de 60 dias (`date_sub(current_timestamp(), 60)`). Dado mais antigo passa no `MERGE` e a Silver fica vazia.


In [0]:
# A célula de parâmetros só cria o schema. Sem saveAsTable a tabela não existe
# e o Silver quebra em FROM bronze.orders.

UFS = ["SP", "RJ", "MG", "RS", "PR", "SC", "BA", "PE", "CE", "GO"]
CATEGORIES = {
    "Mercearia": ["Arroz e feijão", "Óleos", "Massas"],
    "Bebidas": ["Refrigerantes", "Sucos", "Águas"],
    "Limpeza": ["Detergentes", "Desinfetantes"],
    "Higiene": ["Sabonetes", "Shampoos"],
}
BRANDS = ["Marca A", "Marca B", "Marca C", "Genérico"]

def as_string_df(rows, columns):
    """Bronze = texto cru. STRING evita o Spark 'adivinhar' tipo no meio da sujeira."""
    schema = T.StructType([T.StructField(c, T.StringType(), True) for c in columns])
    payload = [
        tuple(None if row.get(c) is None else str(row[c]) for c in columns)
        for row in rows
    ]
    return spark.createDataFrame(payload, schema)

def write_bronze(table, df):
    full_name = fqtn(table)  # `catalogo`.`bronze`.`tabela`
    (
        df.write
          .mode("overwrite")
          .format("delta")
          .option("overwriteSchema", "true")
          .saveAsTable(full_name)
    )
    print(f"gravou {full_name}: {df.count()} linhas")

customers = []
for i in range(1, N_CUSTOMERS + 1):
    uf = random.choice(UFS)
    row = {
        "customer_id": f"C{i:05d}",
        "customer_name": maybe_null(fake.name(), P_EMPTY_FIELDS_CUSTOMER),
        "email": maybe_null(fake.email(), P_EMPTY_FIELDS_CUSTOMER),
        "city": maybe_null(fake.city(), P_EMPTY_FIELDS_CUSTOMER),
        "state": dirty_state(uf),
        "last_update_date": random_date_mixed_formats(random_date_between(50)),
    }
    customers.append(row)
    # mesmo customer_id, outra cidade e outro last_update: a Silver escolhe o mais recente (SCD2)
    if random.random() < P_CUSTOMER_DUP_DIFF_UPDATE:
        dup = dict(row)
        dup["city"] = fake.city()
        dup["last_update_date"] = random_date_mixed_formats(random_date_between(50))
        customers.append(dup)

products = []
cat_names = list(CATEGORIES)
for i in range(1, N_PRODUCTS + 1):
    cat = random.choice(cat_names)
    sub = random.choice(CATEGORIES[cat])
    cost = round(random.uniform(1, 80), 2)
    price = round(cost * random.uniform(1.1, 2.5), 2)
    row = {
        "product_id": f"P{i:05d}",
        "product_name": f"{fake.word().capitalize()} {alnum(4)}",
        "category": cat,
        "subcategory": maybe_null(sub, P_NULL_BRAND_SUBCATEGORY),
        "brand": maybe_null(random.choice(BRANDS), P_NULL_BRAND_SUBCATEGORY),
        "cost_price": maybe_stringify_number(f"{cost:.2f}".replace(".", ",")),
        "list_price": maybe_stringify_number(f"{price:.2f}".replace(".", ",")),
        "is_active": random_bool_inconsistent(),
        "last_update": random_date_mixed_formats(random_date_between(50)),
    }
    products.append(row)
    if random.random() < P_DUP_PRODUCT_ID:
        dup = dict(row)
        dup["list_price"] = f"{price + 1:.2f}".replace(".", ",")
        dup["last_update"] = random_date_mixed_formats(random_date_between(50))
        products.append(dup)

cust_ids = [c["customer_id"] for c in customers]
orders = []
for i in range(1, N_ORDERS + 1):
    dt = random_date_between(50)
    # metade das datas já vem em ISO; a outra metade vem suja para o try_to_timestamp da Silver
    order_date = (
        random_date_mixed_formats(dt)
        if random.random() < P_STRING_DATE_IN_ORDERS
        else dt.strftime("%Y-%m-%d %H:%M:%S")
    )
    amount = round(random.uniform(10, 1500), 2)
    row = {
        "order_id": f"O{i:06d}",
        "customer_id": maybe_null(random.choice(cust_ids), P_NULL_CUSTOMER_IN_ORDERS),
        "order_date": order_date,
        "order_status": random_status_inconsistent(),
        "total_amount": maybe_stringify_number(f"{amount:.2f}".replace(".", ",")),
    }
    orders.append(row)
    if random.random() < P_DUP_ORDERS:
        dup = dict(row)
        dup["order_status"] = random_status_inconsistent()
        dup["order_date"] = random_date_mixed_formats(random_date_between(50))
        orders.append(dup)

order_ids = [o["order_id"] for o in orders]
product_ids = [p["product_id"] for p in products]
items = []
for i in range(1, N_ORDER_ITEMS + 1):
    qty = random.randint(1, 10)
    unit = round(random.uniform(2, 200), 2)
    discount = None if random.random() < P_NULLS_DISCOUNT_PROMO else round(unit * random.uniform(0, 0.3), 2)
    row = {
        "order_item_id": f"I{i:07d}",
        "order_id": random.choice(order_ids),
        "product_id": random.choice(product_ids),
        "quantity": maybe_stringify_number(str(qty)),
        "unit_price": maybe_stringify_number(f"{unit:.2f}".replace(".", ",")),
        "discount_amount": None if discount is None else maybe_stringify_number(f"{discount:.2f}".replace(".", ",")),
        "updated_at": random_date_mixed_formats(random_date_between(50)),
    }
    items.append(row)
    if random.random() < P_DUP_ORDERITEM_SAME_OP:
        dup = dict(row)
        dup["order_item_id"] = f"I{i:07d}B"
        dup["quantity"] = str(qty + 1)
        dup["updated_at"] = random_date_mixed_formats(random_date_between(50))
        items.append(dup)

write_bronze("customers", as_string_df(customers, [
    "customer_id", "customer_name", "email", "city", "state", "last_update_date"
]))
write_bronze("products", as_string_df(products, [
    "product_id", "product_name", "category", "subcategory", "brand",
    "cost_price", "list_price", "is_active", "last_update"
]))
write_bronze("orders", as_string_df(orders, [
    "order_id", "customer_id", "order_date", "order_status", "total_amount"
]))
write_bronze("order_items", as_string_df(items, [
    "order_item_id", "order_id", "product_id", "quantity",
    "unit_price", "discount_amount", "updated_at"
]))

spark.sql(f"SHOW TABLES IN `{CATALOG}`.`bronze`").show(truncate=False)
